# Export the detector's detections: every window, low threshold (GPU)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. With the default settings no cell needs editing, once the training notebook `train-detector` has produced a checkpoint. **Run it once with `PILOT = True` first**: it predicts 6 evenly spaced frames per split (`PILOT_FRAMES`), prints the shapes the library returns, writes the files and reads them back, so the first real run is not the first time the `rfdetr` calls are exercised [VERIFY: nobody has run them on a trained checkpoint yet]. A pilot looks for the checkpoint of a pilot training run (`..._pilot.pth`). `sweep_detector.ipynb` can run its own pilot on a pilot export, since both pick the same 6 holdout frames.

For every frame it predicts each of the 4 windows of `filament.detector_data.frame_windows()` at a score threshold of **0.01**. The threshold is untuned. It is low on purpose so that any later score cut is free: `notebooks/sweep_detector.ipynb` cuts, calibrates and fuses without a GPU. Each mask is mapped to frame coordinates with `filament.detector_data.paste_window_mask`, and stored as pycocotools RLE with its score and window index.

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (train images and train JSON, and the test images for `SPLITS = ("holdout", "test")`).
2. The dataset `ignacioaguirrecandia/solar-code`, a version that contains `filament.detector_export`.
3. The output of the notebook `train-detector` (Add Input, Notebook Output): `detector_<MODEL_CLASS>_<RESOLUTION>.pth` (`..._pilot.pth` from a pilot training run) and `detector_train_manifest.json`. `CHECKPOINT = None` finds it by name under `/kaggle/input`; set `CHECKPOINT` to a path only if it lives elsewhere.

**Settings**

- Accelerator: **GPU T4**, a single T4. Internet: **On** (for `pip install rfdetr[train,loggers]==1.12.0`, the same extras as the training notebook; the pilot of that notebook stopped without them).
- `SPLITS`: `"holdout"` is the date holdout of the U-Net and the detector (never trained on by either). `"test"` is the competition test set: its windows only produce detections. Nothing is fitted, tuned or calibrated on them, and this notebook does not write a submission.

**Output** (`/kaggle/working`): `detections_holdout.jsonl.gz`, `detections_test.jsonl.gz` and `detector_export_manifest.json` (checkpoint file and SHA-256, resolution, threshold, windows, and per split the frames, detections and seconds per image). One line per frame in each file; read it with `filament.detector_export.read_detections`. The files hold masks, so they are not committed.

It does not train, does not write `submission.csv` and does not submit to Kaggle.

## 1. `src` on the path

`src` is the directory that contains `filament/detector_export.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match.

In [ ]:
import os
import sys
from pathlib import Path

MARKER = Path("filament") / "detector_export.py"


def _local_src() -> Path | None:
    """A checkout that already has the module this notebook needs (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / MARKER).is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    The marker may sit at any depth. Directories named ``competitions`` are not
    walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / MARKER).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the private
    dataset's ``src`` is used in place. It is not copied to ``/kaggle/working``: a
    copy there would be saved with this notebook's output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            f"Could not find src/{MARKER.as_posix()} in the working directory or "
            "under /kaggle/input (any depth outside competitions). Attach the "
            "dataset ignacioaguirrecandia/solar-code. If it is attached, its "
            "version predates the detector modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Settings

`MODEL_CLASS` and `RESOLUTION` must be the ones the checkpoint was trained with: the training manifest next to the checkpoint is compared with them, and so is the file's SHA-256. A checkpoint from a pilot training run is refused unless this run is a pilot too.

In [ ]:
# ---- Settings. Nothing below this cell needs editing. -----------------------
PILOT = False  # True: PILOT_FRAMES evenly spaced frames of each split, and a pilot checkpoint
# None: search /kaggle/input for detector_<MODEL_CLASS>_<RESOLUTION>[_pilot].pth.
# Otherwise the path of the .pth file.
CHECKPOINT = None
MODEL_CLASS = "RFDETRSegLarge"
RESOLUTION = 768
SPLITS = ("holdout", "test")
SCORE_THRESHOLD = 0.01  # untuned; low on purpose, so later cuts are free
RFDETR_VERSION = "1.12.0"
PILOT_FRAMES = 6  # per split

## 3. GPU, `rfdetr` and the checkpoint

With `CHECKPOINT = None`, or if the file at `CHECKPOINT` does not exist, `/kaggle/input` is searched at any depth for exactly one `detector_<MODEL_CLASS>_<RESOLUTION>.pth` (`..._pilot.pth` when `PILOT` is True). A training manifest next to it must agree on the SHA-256, the model class and the resolution.

In [ ]:
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"rfdetr[train,loggers]=={RFDETR_VERSION}",
    ],
    check=True,
)

In [ ]:
import json
import time
from importlib import metadata

import numpy as np
import torch

from filament.detector_train import sha256_file

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Set the accelerator to GPU T4 and run again.")
print(f"gpu={torch.cuda.get_device_name(0)} torch={torch.__version__}")
if metadata.version("rfdetr") != RFDETR_VERSION:
    raise RuntimeError(
        f"rfdetr {metadata.version('rfdetr')} is installed, not {RFDETR_VERSION}"
    )


def _find_checkpoint() -> Path:
    name = f"detector_{MODEL_CLASS}_{RESOLUTION}{'_pilot' if PILOT else ''}.pth"
    wanted = Path(CHECKPOINT) if CHECKPOINT else Path("/kaggle/input") / name
    if CHECKPOINT and wanted.is_file():
        return wanted
    matches = []
    for dirpath, dirnames, filenames in os.walk("/kaggle/input", topdown=True):
        dirnames[:] = [d for d in dirnames if d != "competitions"]
        if name in filenames:
            matches.append(Path(dirpath) / name)
    if len(matches) != 1:
        raise FileNotFoundError(
            f"{wanted} does not exist and {len(matches)} files named {name} were found "
            f"under /kaggle/input: {matches}. Attach the output of train-detector."
        )
    return matches[0]


CHECKPOINT_PATH = _find_checkpoint()
CHECKPOINT_SHA256 = sha256_file(CHECKPOINT_PATH)
print(f"checkpoint={CHECKPOINT_PATH} sha256={CHECKPOINT_SHA256}")
TRAIN_MANIFEST = CHECKPOINT_PATH.with_name("detector_train_manifest.json")
if TRAIN_MANIFEST.is_file():
    trained = json.loads(TRAIN_MANIFEST.read_text(encoding="utf-8"))
    for key, mine in (
        ("checkpoint_sha256", CHECKPOINT_SHA256),
        ("model_class", MODEL_CLASS),
        ("resolution", RESOLUTION),
    ):
        if trained.get(key) != mine:
            raise RuntimeError(
                f"training manifest {key}={trained.get(key)!r}, this run has {mine!r}"
            )
    if trained.get("pilot") and not PILOT:
        raise RuntimeError(
            "This checkpoint came from a pilot training run. Train it for real first."
        )
    print(
        f"training manifest agrees: source={trained['checkpoint_source']} "
        f"epochs requested={trained['epochs_requested']} stopped_by_guard={trained['stopped_by_guard']}"
    )
else:
    print(
        "WARNING: no detector_train_manifest.json next to the checkpoint; its hash and settings are unchecked."
    )

## 4. Load the model

`RFDETR.from_checkpoint` restores the model, and `.inference()` makes the predictor the library recommends for speed (its warning says a model used straight from training is slower). If `.inference()` does not take the call as written, the model's own `predict` is used and the notebook says so [VERIFY on the pilot].

In [ ]:
from rfdetr import RFDETR

model = RFDETR.from_checkpoint(str(CHECKPOINT_PATH))
restored = getattr(getattr(model, "model_config", None), "resolution", None)
if restored is None:
    print("WARNING: could not read the resolution back from the loaded model [VERIFY].")
elif restored != RESOLUTION:
    raise RuntimeError(
        f"The checkpoint runs at resolution {restored}, not {RESOLUTION}."
    )
else:
    print(f"model restored at resolution {restored}")
try:
    predictor = model.inference()
    print("using model.inference()")
except (AttributeError, TypeError) as error:
    predictor = model
    print(f"model.inference() is not usable here ({error!r}); using model.predict")

## 5. Frames and windows

`holdout` is `build_detector_splits(...).holdout`, in its order. `test` is every `.jpeg` of the competition's `test_images`, sorted by name. The 4 windows are the ones the detector was trained on, and each is cut, resized and passed through the same JPEG round trip as the training files (`filament.detector_export.window_input`).

In [ ]:
from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.crop_data import read_gray
from filament.detector_data import build_detector_splits, frame_windows
from filament.detector_export import (
    EXPORT_MANIFEST,
    DetectionWriter,
    detections_name,
    frame_detections,
    read_detections,
    window_input,
)
from filament.detector_train import subsample_evenly

IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
TEST_IMAGES = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026/test/test_images"
)
OUTPUT_DIR = (
    Path("/kaggle/working")
    if Path("/kaggle/working").is_dir()
    else Path.cwd() / "detector_export"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FULL_SIDE = 2048
WINDOWS = frame_windows()
unknown = sorted(set(SPLITS) - {"holdout", "test"})
if unknown:
    raise ValueError(f"unknown split(s) {unknown}")


def frames_of(split):
    if split == "holdout":
        loaded = load_train_annotations(Path(TRAIN_ANNOTATIONS_PATH))
        holdout = build_detector_splits(loaded).holdout
        return [
            (image.file_name, IMAGE_DIR / image.file_name, (image.height, image.width))
            for image in holdout
        ]
    paths = sorted(
        p for p in TEST_IMAGES.iterdir() if p.is_file() and p.suffix.lower() == ".jpeg"
    )
    if not paths:
        raise FileNotFoundError(
            f"No .jpeg files in {TEST_IMAGES}. Attach the competition."
        )
    return [(p.name, p, (FULL_SIDE, FULL_SIDE)) for p in paths]


FRAMES = {split: frames_of(split) for split in SPLITS}
if PILOT:
    # Evenly spaced, the same rule as the pilot of sweep_detector.ipynb, so that
    # notebook can run on a pilot export.
    FRAMES = {
        split: list(subsample_evenly(frames, PILOT_FRAMES))
        for split, frames in FRAMES.items()
    }
print(
    {split: len(frames) for split, frames in FRAMES.items()}, f"windows={len(WINDOWS)}"
)

## 6. Predict

One `predict` call per window, at `SCORE_THRESHOLD`. `inference_seconds` is the time of the `predict` calls (with the GPU synchronized); `total_seconds` also counts the window preparation and the mapping to RLE. Both are per frame. The first frame of the first split prints what the library returned, to check the assumed `Detections` fields (`mask`, `confidence`).

In [ ]:
def predict_window(rgb):
    result = predictor.predict(rgb, threshold=SCORE_THRESHOLD)
    if isinstance(result, (list, tuple)):
        result = result[0]
    masks = getattr(result, "mask", None)
    scores = getattr(result, "confidence", None)
    if masks is None or scores is None or len(scores) == 0:
        return np.zeros((0, RESOLUTION, RESOLUTION), dtype=bool), np.zeros(0)
    return np.asarray(masks), np.asarray(scores, dtype=np.float64)


announced = False
split_stats = {}
for split in SPLITS:
    frames = FRAMES[split]
    path = OUTPUT_DIR / detections_name(split)
    inference_seconds = 0.0
    empty_masks = 0
    frames_without = 0
    begin = time.monotonic()
    with DetectionWriter(path) as writer:
        for index, (file_name, image_path, shape) in enumerate(frames, start=1):
            gray = read_gray(image_path, shape)
            found = []
            for window_index, window in enumerate(WINDOWS):
                rgb = window_input(gray, window, RESOLUTION)
                torch.cuda.synchronize()
                started = time.monotonic()
                masks, scores = predict_window(rgb)
                torch.cuda.synchronize()
                inference_seconds += time.monotonic() - started
                if not announced:
                    announced = True
                    print(
                        f"first window: masks {masks.shape} {masks.dtype}, scores {scores.shape} "
                        f"(max {scores.max() if scores.size else float('nan'):.3f})"
                    )
                if masks.size and masks.shape[1:] != (RESOLUTION, RESOLUTION):
                    print(
                        f"note: masks come back at {masks.shape[1:]}, resized to the window"
                    )
                detections, empty = frame_detections(
                    masks, scores, window_index, window, shape, SCORE_THRESHOLD
                )
                empty_masks += empty
                found.extend(detections)
            frames_without += not found
            writer.write_frame(file_name, shape, found)
            if index == 1 or index % 20 == 0 or index == len(frames):
                print(
                    f"  {split} {index}/{len(frames)} {file_name}: {len(found)} detections",
                    flush=True,
                )
    total = time.monotonic() - begin
    # Read the file back: the sweep notebook trusts it, and the writer's own
    # counters would not show a truncated or unreadable file.
    names_back = []
    detections_back = 0
    for file_name, shape, found_back in read_detections(path):
        names_back.append(file_name)
        detections_back += len(found_back)
        for detection in found_back:
            top, left, bottom, right = detection.bbox
            if not (
                detection.shape == shape
                and detection.score >= SCORE_THRESHOLD - 1e-6
                and 0 <= top < bottom <= shape[0]
                and 0 <= left < right <= shape[1]
            ):
                raise RuntimeError(
                    f"{path.name}: {file_name} holds a detection with bbox "
                    f"{detection.bbox}, score {detection.score}, shape {detection.shape}"
                )
    if names_back != [name for name, _, _ in frames] or (
        len(names_back),
        detections_back,
    ) != (writer.frames, writer.detections):
        raise RuntimeError(
            f"{path.name} read back as {len(names_back)} frames and {detections_back} "
            f"detections; {writer.frames} and {writer.detections} were written"
        )
    print(
        f"  read back {path.name}: {len(names_back)} frames, {detections_back} detections"
    )
    split_stats[split] = {
        "frames": writer.frames,
        "detections": writer.detections,
        "frames_without_detections": frames_without,
        "empty_masks_skipped": empty_masks,
        "inference_seconds_per_image": round(
            inference_seconds / max(writer.frames, 1), 4
        ),
        "total_seconds_per_image": round(total / max(writer.frames, 1), 4),
        "file": path.name,
    }
    print(split, split_stats[split])

## 7. Manifest

The manifest sits next to the detection files. `notebooks/sweep_detector.ipynb` looks for a directory that has both `detections_<split>.jsonl.gz` and this manifest, and checks the window geometry in it.

In [ ]:
manifest = {
    "format_version": 1,
    "pilot": PILOT,
    "checkpoint_file": CHECKPOINT_PATH.name,
    "checkpoint_sha256": CHECKPOINT_SHA256,
    "model_class": MODEL_CLASS,
    "resolution": RESOLUTION,
    "score_threshold": SCORE_THRESHOLD,
    "rfdetr": RFDETR_VERSION,
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "windows": [[w.x0, w.y0, w.x1, w.y1] for w in WINDOWS],
    "splits": split_stats,
}
(OUTPUT_DIR / EXPORT_MANIFEST).write_text(
    json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8"
)
print(f"wrote {OUTPUT_DIR / EXPORT_MANIFEST}")
print(
    "Test windows only produced detections: nothing was fitted on them. No submission was written or made."
)